[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLManuals/Deep-Learning/blob/main/chapter08_tensors_in_pytorch/notebook.ipynb)

# Chapter 8: Tensors in PyTorch

Run the cell below once, first, to install any missing packages (only needed on Google Colab; skip it if you're running locally and already have them).

In [ ]:
%pip install -q torch numpy scikit-learn

Part 2 ended with a fair question. You have already written a working neural network in numpy, derived its gradients yourself and watched it learn XOR, so why learn a framework at all?

The people who built PyTorch asked a version of that question too, and their answer has an odd twist in it. Through the early 2010s the most loved toolkit for neural network research was called Torch, written around 2010 by Ronan Collobert, Clement Farabet and Koray Kavukcuoglu. Researchers liked it. It was fast, it was flexible, and it did what they wanted. It had exactly one serious problem, which was that you had to write it in a language called Lua.

Lua was not hard to learn, but almost nobody in the wider research world used it, while nearly everyone was already writing Python. In early 2016 a group of Torch contributors, among them Soumith Chintala, began rebuilding the whole thing around Python, and a student in Poland named Adam Paszke volunteered to help replace the Lua layer. The result was released in 2017 as **PyTorch**, and it spread through the research community within months. A framework that was already good had been held back, for years, by nothing more than the language it spoke.

PyTorch kept one design choice from Torch that mattered enormously, which is that your code runs line by line like ordinary Python, rather than being assembled into a fixed plan before anything executes. That is why you can put a `print` inside a network and have it work, and it is also what makes Chapter 9 possible. Everything in PyTorch is built on one object, the **tensor**, and this chapter is about that object. Before you read on, try to guess the answers to these four questions.

1. **What could a framework give you that your numpy network does not already have?**
2. **If a tensor is basically an array, why give it a new name?**
3. **What might go wrong when you move data between numpy and PyTorch?**
4. **Why would a framework care which kind of number you store, whole or decimal?**

## Meet PyTorch

Start with the honest answer to the question Part 2 left open. Your numpy network was real, and nothing about it was fake or simplified. A framework gives you three things on top of it, and Figure 8.1 shows them.

> 🖼️ **Figure 8.1:** What a framework adds to the network you already built. All three rest on one object, which is the tensor.

None of those three replace what you learned. They remove work you now understand well enough to hand over safely, which is exactly the right moment to hand it over. If you have not installed PyTorch, the command is `pip install torch`, and the first cell checks that it is there.

In [ ]:
import torch

print(torch.__version__)

Your version number will differ, and the `cpu` on the end means this copy has no GPU support, which is fine until Chapter 19. **PyTorch does not replace what you learned in Part 2, it automates it, and the thing it automates is the one piece of a network that is tedious rather than interesting.**

## The Tensor, a Smarter Array

A **tensor** is PyTorch's array, and if you are comfortable with a numpy array you already understand most of it, because the two were deliberately made to look alike. The word means a grid of numbers with any number of dimensions, and Figure 8.2 shows the ladder you will meet.

> 🖼️ **Figure 8.2:** Tensors of increasing dimension. A colour photograph is a three dimensional tensor, and a batch of photographs is a four dimensional one.

You build a tensor from ordinary Python lists, and it reports its size and the kind of number it holds.

In [ ]:
t = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
print(t)
print(t.shape, t.dtype)

The `shape` says two rows and three columns, exactly as in Chapter 3. The `dtype` is the detail worth pausing on, because it catches beginners constantly. PyTorch looked at your numbers, saw decimal points, and chose **float32**, meaning decimals stored in 32 bits. Had you written the same numbers without decimal points it would have chosen whole numbers instead.

In [ ]:
print(torch.tensor([1, 2, 3]).dtype)
print(torch.tensor([1.0, 2.0, 3.0]).dtype)

That difference matters. Networks learn by making tiny adjustments, and a whole number cannot hold one, so a tensor of the wrong type will fail outright when Chapter 9 asks it for gradients. **Weights, inputs and targets in a network are almost always float32, so write your numbers with decimal points and check `dtype` whenever something behaves strangely.**

PyTorch also builds common tensors for you without a list. The seed makes the random ones reproducible, exactly as `default_rng` did in numpy.

In [ ]:
torch.manual_seed(7)
print(torch.zeros(2, 3))
print(torch.arange(0, 10, 2))
print(torch.randn(2, 3).round(decimals=3))

The last one draws from the bell shaped spread that started the XOR network in Chapter 7.

## Everyday Tensor Operations

Arithmetic on tensors works element by element, so adding two tensors adds matching positions, and multiplying by a single number scales everything.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
y = torch.tensor([10.0, 20.0, 30.0])
print(x + y, x * y, x * 2)

Notice that `x * y` multiplied position by position rather than doing anything matrix like. That is a different operation from the `@` you used for layers, and confusing the two is one of the most common bugs in deep learning code.

When shapes do not match, PyTorch stretches the smaller one to fit, which is called **broadcasting**. Adding a row of two numbers to a table with two columns adds it to every row, and that is precisely how a layer adds its biases.

In [ ]:
m = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])
print(m + torch.tensor([10.0, 100.0]))

Figure 8.3 shows what happened. The single row was copied down to meet every row of the table, without ever actually being stored twice.

> 🖼️ **Figure 8.3:** Broadcasting. A single row meets a table by being reused for every row, which is how a layer adds one bias per neuron to a whole batch at once.

The `@` symbol does the matrix multiplication from Chapter 3. The next cell runs the three teachers from that chapter, with their weight matrix and the student who studied 8 hours and slept 3.

In [ ]:
W = torch.tensor([[0.9, 0.1],
                  [0.1, 0.9],
                  [0.5, 0.5]])
student = torch.tensor([8.0, 3.0])
print(W @ student)

Those are the same weighted sums as in Chapter 3, before the biases of $-4$, $-4$ and $-5$ were subtracted to give 3.5, $-0.5$ and 0.5. Changing a tensor's shape without changing its numbers is also something you will do constantly, usually to line up a layer's input.

In [ ]:
r = torch.arange(6.0)
print(r)
print(r.reshape(2, 3))
print(r.reshape(2, 3).T)

Finally, tensors can collapse themselves down to summaries, which is how every loss in Chapter 5 was computed.

In [ ]:
s = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])
print(s.sum(), s.mean(), s.sum(dim=0), s.max())

The `dim=0` argument means collapse down the rows, leaving one number per column. Without it, everything collapses to a single number. Table 8.1 gathers the translations you will reach for most often.

**Table 8.1: The same operations in both libraries. The similarity is deliberate, so that numpy habits carry straight over.**

| What you want | numpy | PyTorch |
|---|---|---|
| Build from a list | `np.array([1.0, 2.0])` | `torch.tensor([1.0, 2.0])` |
| Its size | `a.shape` | `t.shape` |
| A grid of zeros | `np.zeros((2, 3))` | `torch.zeros(2, 3)` |
| Random bell shaped | `rng.normal(0, 1, (2, 3))` | `torch.randn(2, 3)` |
| Matrix multiply | `A @ b` | `A @ b` |
| Change shape | `a.reshape(2, 3)` | `t.reshape(2, 3)` |
| Average of all | `a.mean()` | `t.mean()` |

## From numpy to Tensors

Real projects mix the two libraries constantly, because data arrives through pandas and numpy and then has to become tensors. Moving in either direction is one call.

In [ ]:
import numpy as np

arr = np.array([1.0, 2.0, 3.0])
t = torch.from_numpy(arr)
print(t, t.dtype)

Look at that `dtype`. numpy defaults to float64, PyTorch defaults to float32, and `from_numpy` kept numpy's choice. Mixing the two types in one network produces an error about expected types that confuses almost everyone the first time.

There is a second surprise hiding in that cell, and it is sharper. The tensor and the array share the same memory, so changing one changes the other.

In [ ]:
arr[0] = 99.0
print(t)

Nobody touched `t`, and `t` changed. That sharing makes conversion instant for huge datasets, but it causes bugs that are genuinely hard to find. Passing the array to `torch.tensor` instead makes a real copy and lets you fix the type at the same time, which is what you usually want.

In [ ]:
arr2 = np.array([1.0, 2.0, 3.0])
t2 = torch.tensor(arr2, dtype=torch.float32)
arr2[0] = 99.0
print(t2)

**Use `torch.from_numpy` when you want speed and know the two will share memory, and `torch.tensor` when you want a safe copy with the type you choose.**

Now put it together. The next cells rebuild the XOR network from Chapter 3, the one whose weights you chose by hand, using nothing but tensors.

In [ ]:
def step(z):
    return (z > 0).float()

The weights are the ones you reasoned out in Chapter 3, now written as tensors.

In [ ]:
W1 = torch.tensor([[ 1.0,  1.0],
                   [-1.0, -1.0]])
b1 = torch.tensor([-0.5, 1.5])
W2 = torch.tensor([1.0, 1.0])
b2 = torch.tensor(-1.5)

In [ ]:
for pair in ([0., 0.], [0., 1.], [1., 0.], [1., 1.]):
    x = torch.tensor(pair)
    h = step(W1 @ x + b1)
    y = step(W2 @ h + b2)
    print(pair, h.numpy(), y.item())

Those are the same four answers, and the same hidden values, as Chapter 3 printed in numpy. The code barely changed, which is the point of this chapter. `.item()` pulls a single number out of a tensor as an ordinary Python number, and `.numpy()` converts a whole tensor back, both of which you will use whenever you want to print or plot a result.

Everything so far has been housekeeping, and none of it is why anyone uses PyTorch. The reason is in the next chapter.

## Practice and Review

Reading about deep learning is only half of learning it. Pulling an idea back out of your memory makes it stick much better than reading it again, so this section gives you two kinds of practice, one for your hands and one for your memory.

### Try It Yourself

1. **Break it on purpose.** Build `torch.tensor([1, 2, 3])` and try `t / 2` and then `t.mean()`. Read the error that `mean` gives, and fix it by adding decimal points.
2. **Add the biases.** Extend the three teachers cell with `b = torch.tensor([-4.0, -4.0, -5.0])`, print `W @ student + b`, and check you get the same values as Chapter 3.
3. **Catch the sharing.** Make an array, convert it with `torch.from_numpy`, change the tensor this time rather than the array, and confirm the array changed too.

### Check Your Understanding

Try to answer each question from memory before you look at the answers that follow.

1. **What three things does PyTorch add to a network you could write in numpy?**
2. **Why do network weights need to be decimals rather than whole numbers?**
3. **What is the difference between `x * y` and `x @ y`?**
4. **What does broadcasting do, and where does a network rely on it?**
5. **What is the trap in `torch.from_numpy`?**

### Answers

1. Gradients worked out automatically, ready made layers and optimizers, and the ability to run on a GPU.
2. Because learning means making tiny adjustments, and a whole number cannot hold a tiny adjustment.
3. The first multiplies matching positions one by one, while the second does matrix multiplication, which is what a layer does.
4. It stretches a smaller shape to meet a larger one, which is how a layer adds one bias per neuron across a whole batch.
5. The tensor and the array share memory, so changing either one changes the other, and the type stays numpy's float64.

## What Comes Next

You can now build tensors, check their shape and type, do arithmetic and matrix multiplication with them, and move data between numpy and PyTorch without being caught out.

- **Chapter 9, Autograd: Calculus for Free,** makes tensors remember what was done to them, so PyTorch can work out every gradient by itself.
- **Chapter 10, Building a Network with nn.Module,** replaces your loose weight matrices with a proper, reusable model.

Chapter 9 opens by asking PyTorch for the gradients you worked out by hand in Chapter 7, to see whether it agrees with you.